# Кольцевые коммуникации

Реализуйте три операции на CPU с Gloo. 
- `rank` — номер процесса,
- `world_size` — число процессов.
- Входные тензоры — одномерные CPU-тензоры одинакового размера на всех процессах. Их менять нельзя

In [1]:
%%writefile collectives.py
import torch
import torch.distributed as dist


def ring_reduce_scatter(tensor):
    """Сложить тензоры по кольцу и вернуть часть суммы с индексом rank."""
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    if tensor.ndim != 1 or tensor.device.type != "cpu":
        raise ValueError("Ожидается одномерный CPU-тензор.")
    if tensor.numel() % world_size != 0:
        raise ValueError("Размер тензора должен делиться на число процессов.")
    if tensor.numel() == 0:
        return tensor.clone()
    chunks = [chunk.clone() for chunk in tensor.chunk(world_size)]

    next_rank = (rank + 1) % world_size
    prev_rank = (rank - 1) % world_size

    # Сдвиг на один чанк выбран так, чтобы полная сумма осталась в chunks[rank].
    for step in range(world_size - 1):
        send_index = (rank - step - 1) % world_size
        recv_index = (rank - step - 2) % world_size
        received = torch.empty_like(chunks[recv_index])

        requests = dist.batch_isend_irecv([
            dist.P2POp(dist.isend, chunks[send_index], next_rank),
            dist.P2POp(dist.irecv, received, prev_rank),
        ])
        # Обе операции запущены до ожидания; отправляемый буфер пока не меняем.
        for request in requests:
            request.wait()
        chunks[recv_index].add_(received)

    return chunks[rank]


def ring_all_gather(shard):
    """Собрать части по порядку rank в полный тензор на каждом процессе."""
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    if shard.ndim != 1 or shard.device.type != "cpu":
        raise ValueError("Ожидается одномерный CPU-тензор.")
    chunks = [torch.empty_like(shard) for _ in range(world_size)]
    chunks[rank].copy_(shard)
    if shard.numel() == 0:
        return torch.cat(chunks)

    next_rank = (rank + 1) % world_size
    prev_rank = (rank - 1) % world_size

    for step in range(world_size - 1):
        # Сначала отправляем свою часть, затем часть, полученную на прошлом шаге.
        send_index = (rank - step) % world_size
        recv_index = (rank - step - 1) % world_size
        requests = dist.batch_isend_irecv([
            dist.P2POp(dist.isend, chunks[send_index], next_rank),
            dist.P2POp(dist.irecv, chunks[recv_index], prev_rank),
        ])
        for request in requests:
            request.wait()

    return torch.cat(chunks)


def ring_all_reduce(tensor):
    """Вернуть полную сумму на каждом процессе, не изменяя входной тензор."""
    return ring_all_gather(ring_reduce_scatter(tensor))


Overwriting collectives.py


## Сравнение с PyTorch

Сравним нашу реализацию с готовыми функциями торча `dist.reduce_scatter_tensor`, `dist.all_gather`
и `dist.all_reduce` на нескольких размерах тензоров. 

In [2]:
%%writefile check.py
from datetime import timedelta

import torch
import torch.distributed as dist

from collectives import ring_all_gather, ring_all_reduce, ring_reduce_scatter


torch.set_num_threads(1)
dist.init_process_group("gloo", timeout=timedelta(seconds=30))
try:
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    generator = torch.Generator().manual_seed(42 + rank)

    for chunk_size in (1, 7, 31):
        values = torch.randn(world_size * chunk_size, generator=generator, dtype=torch.float64)
        original = values.clone()
        expected_shard = torch.empty(chunk_size, dtype=values.dtype)
        dist.reduce_scatter_tensor(expected_shard, values.clone(), op=dist.ReduceOp.SUM)
        torch.testing.assert_close(ring_reduce_scatter(values), expected_shard)
        torch.testing.assert_close(values, original, rtol=0, atol=0)

        shard = torch.randn(chunk_size, generator=generator, dtype=torch.float64)
        original_shard = shard.clone()
        expected_parts = [torch.empty_like(shard) for _ in range(world_size)]
        dist.all_gather(expected_parts, shard)
        torch.testing.assert_close(ring_all_gather(shard), torch.cat(expected_parts))
        torch.testing.assert_close(shard, original_shard, rtol=0, atol=0)

        expected_sum = values.clone()
        dist.all_reduce(expected_sum, op=dist.ReduceOp.SUM)
        torch.testing.assert_close(ring_all_reduce(values), expected_sum)
        torch.testing.assert_close(values, original, rtol=0, atol=0)

    if rank == 0:
        print(f"OK: все три операции прошли проверки на {world_size} процессах.")
finally:
    dist.destroy_process_group()

Writing check.py


In [3]:
import sys

!{sys.executable} -m torch.distributed.run --standalone --nproc-per-node=2 check.py

/private/tmp/hw2_communications_env/lib/python3.11/site-packages/torch/_subclasses/functional_tensor.py:275: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:81.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


W1013 00:00:04.053000 41718 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.

*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
*****************************************


/private/tmp/hw2_communications_env/lib/python3.11/site-packages/torch/_subclasses/functional_tensor.py:275: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:81.)
  cpu = _conversion_method_template(device=torch.device("cpu"))
/private/tmp/hw2_communications_env/lib/python3.11/site-packages/torch/_subclasses/functional_tensor.py:275: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:81.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


OK: все три операции прошли проверки на 2 процессах.


In [4]:
!{sys.executable} -m torch.distributed.run --standalone --nproc-per-node=4 check.py

/private/tmp/hw2_communications_env/lib/python3.11/site-packages/torch/_subclasses/functional_tensor.py:275: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:81.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


W1013 00:00:17.307000 41767 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.

*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
*****************************************


/private/tmp/hw2_communications_env/lib/python3.11/site-packages/torch/_subclasses/functional_tensor.py:275: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:81.)
  cpu = _conversion_method_template(device=torch.device("cpu"))
/private/tmp/hw2_communications_env/lib/python3.11/site-packages/torch/_subclasses/functional_tensor.py:275: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:81.)
  cpu = _conversion_method_template(device=torch.device("cpu"))
/private/tmp/hw2_communications_env/lib/python3.11/site-packages/torch/_subclasses/functional_tensor.py:275: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:81.)
  cpu = _conversion_method_te

OK: все три операции прошли проверки на 4 процессах.
